# Inference và tạo submission

Notebook tải checkpoint YOLO26m và fastText đã train, dự đoán trên `data/test/images` và `data/intent_test.csv`, sau đó ghi một file có đúng định dạng `sample_submission.csv`. Không dùng ensemble hoặc TTA.

In [1]:
# Cài dependency nếu cần:
# %pip install -U ultralytics fasttext-wheel
from pathlib import Path
import json
import re
import pandas as pd
from ultralytics import YOLO
import fasttext

ROOT = Path.cwd()
DATA = ROOT / 'data'
if not DATA.exists():
    for parent in ROOT.parents:
        if (parent / 'data').exists():
            ROOT, DATA = parent, parent / 'data'
            break

YOLO_WEIGHTS = ROOT / 'runs' / 'yolo26m_objects' / 'weights' / 'best.pt'
FASTTEXT_MODEL = ROOT / 'fasttext_work' / 'intent_fasttext_3gram_200k.bin'
SAMPLE_PATH = DATA / 'sample_submission.csv'
OUT_PATH = ROOT / 'submission.csv'

assert YOLO_WEIGHTS.exists(), f'Missing YOLO checkpoint: {YOLO_WEIGHTS}'
assert FASTTEXT_MODEL.exists(), f'Missing fastText model: {FASTTEXT_MODEL}'
sample = pd.read_csv(SAMPLE_PATH)
test_images = DATA / 'test' / 'images'
intent_test = pd.read_csv(DATA / 'intent_test.csv')
print(sample.shape, sample.task_type.value_counts().to_dict())

(8000, 3) {'DETECTION': 4000, 'INTENT': 4000}


## Inference detection

Tọa độ YOLO được chuyển sang `[x1, y1, x2, y2]` theo pixel ảnh, đúng format mẫu.

In [2]:
detector = YOLO(str(YOLO_WEIGHTS))
det_rows = {}

for result in detector.predict(
    source=str(test_images),
    imgsz=640,
    conf=0.25,
    iou=0.7,
    max_det=100,
    device=0,
    stream=True,
    verbose=False,
):
    sample_id = Path(result.path).stem
    boxes = []
    if result.boxes is not None:
        for xyxy, conf, cls in zip(result.boxes.xyxy.cpu().tolist(), result.boxes.conf.cpu().tolist(), result.boxes.cls.cpu().tolist()):
            boxes.append({
                'class_id': int(cls),
                'score': round(float(conf), 6),
                'bbox': [round(float(v), 2) for v in xyxy],
            })
    det_rows[sample_id] = json.dumps(boxes, separators=(',', ':'))

print('detection predictions:', len(det_rows))

detection predictions: 4000


## Inference intent

In [3]:
def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z0-9_\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

intent_model = fasttext.load_model(str(FASTTEXT_MODEL))
intent_rows = {}
for _, row in intent_test.iterrows():
    labels, scores = intent_model.predict(clean_text(row['question']), k=1)
    intent_rows[row['question_id']] = labels[0].replace('__label__', '')
print('intent predictions:', len(intent_rows))

intent predictions: 4000


## Gộp theo đúng thứ tự sample_submission

In [4]:
def make_prediction(row):
    if row.task_type == 'DETECTION':
        return det_rows.get(row.sample_id, '[]')
    if row.task_type == 'INTENT':
        return intent_rows[row.sample_id]
    raise ValueError(f'Unknown task_type: {row.task_type}')

submission = sample.copy()
submission['prediction'] = submission.apply(make_prediction, axis=1)
assert len(submission) == 8000
assert submission[['task_type', 'sample_id']].duplicated().sum() == 0
assert set(submission.loc[submission.task_type == 'DETECTION', 'sample_id']) == set(det_rows)
assert set(submission.loc[submission.task_type == 'INTENT', 'sample_id']) == set(intent_rows)
submission.to_csv(OUT_PATH, index=False)
print(f'Saved: {OUT_PATH}')
print(submission.head(3).to_string(index=False))
print(submission.tail(3).to_string(index=False))

Saved: c:\Users\admin\OneDrive - The University of Technology\Desktop\edge_ai\submission.csv
task_type sample_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                prediction
DETECTION  img_0001                                                                                                                                                                                                                                                                              [{"class_id":5,"score":0.950523,"bbox":[56.64,72.56,106.4,116.86]},{"class_id":2,"score":0.941034,"bbox":[133.58,100.26,189.